# BIND core fine-tune validation: does `--core_weight` sharpen the cores?

Compares **baseline** (`fm_two_head` epoch047) vs the **core-weighted fine-tune**
(`fm_two_head_core`) against truth, on held-out test patches. The core-weighted loss
up-weights the sparse co-located cores so the flow stops smoothing them — the high-k
P(k) fix. We check it worked **and** that goals 1–4 survived:

1. **High-k:** per-channel patch transfer `T(k)=sqrt(P_gen/P_truth)` — does `T_stars`/`T_total` rise toward 1?
2. **Core sharpness:** BCG-centered radial profile + central-pixel concentration.
3. **Goal 1 (mass):** per-halo integrated mass, gen vs truth.
4. **Goal 3 (bulk):** pixel-value distributions + KS — did core-weighting degrade the smooth field?
5. **Visual:** stellar cores, truth / baseline / core, side by side.

Channels: `DM_hydro, Gas, Stars` · box 6.25 Mpc/h · 128². Needs a GPU (sampling).

In [ ]:
import sys, os
sys.path.insert(0, '/mnt/home/mlee1/vdm_bind2')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
import numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from bind.data import load_file_list, AstroDataset, NormStats
from bind.train import FlowMatchingLit
from bind.metrics import power_spectrum_2d, radial_profile, compute_mass, CHANNEL_NAMES
from bind.inference.pipeline import _denormalize_to_physical

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
print('Device:', device)

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
DATA_ROOT = '/mnt/home/mlee1/ceph/train_data_rotated2_128_cpu'
RUNS_DIR  = Path('/mnt/home/mlee1/ceph/fm_runs')
BOX_SIZE, N_PIX = 6.25, 128
PIX = BOX_SIZE / N_PIX

# baseline = the better checkpoint (epoch047); core = the fine-tune (latest last.ckpt)
RUNS = {
    'baseline': (RUNS_DIR / 'fm_two_head',      'epoch047-val_loss0.2138.ckpt'),
    'core':     (RUNS_DIR / 'fm_two_head_core', 'last.ckpt'),
}
COLORS = {'baseline': 'tab:orange', 'core': 'tab:blue', 'truth': 'k'}

N_TEST, N_STEPS, BATCH, N_WORKERS, SEED = 200, 50, 64, 8, 42
rng = np.random.RandomState(SEED)
all_test = load_file_list(DATA_ROOT, 'test')
test_files = [all_test[i] for i in rng.choice(len(all_test), min(N_TEST, len(all_test)), replace=False)]
print(f'{len(test_files)} test patches')

In [ ]:
# ── Helpers ─────────────────────────────────────────────────────────────────
def load_ckpt(run_dir, ckpt_name):
    p = Path(run_dir) / 'checkpoints' / ckpt_name
    model = FlowMatchingLit.load_from_checkpoint(str(p), map_location=device).eval().to(device)
    ns = NormStats.load(Path(run_dir) / 'norm_stats.npz')
    return model, ns

@torch.no_grad()
def generate(model, ns, files):
    ds = AstroDataset(files, ns)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=N_WORKERS,
                        pin_memory=True, persistent_workers=(N_WORKERS > 0))
    real, gen, params, mass = [], [], [], []
    for b in tqdm(loader, desc='sampling', leave=False):
        cond, ls, pr = b['condition'].to(device), b['large_scale'].to(device), b['params'].to(device)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            g = model.fm.sample(cond, ls, pr, n_steps=N_STEPS)
        real.append(_denormalize_to_physical(b['target'].numpy().copy(), ns))
        gen.append(_denormalize_to_physical(g.float().cpu().numpy(), ns))
        params.append(b['params'].numpy())
        if 'halo_mass' in b: mass.append(b['halo_mass'].numpy())
    return (np.concatenate(real), np.concatenate(gen), np.concatenate(params),
            np.concatenate(mass) if mass else None)

def stacked_pk(fields):
    pks, k = [], None
    for f in fields:
        if f.mean() <= 0: continue
        k, p = power_spectrum_2d(f, box_size=BOX_SIZE); pks.append(p)
    return k, np.mean(pks, 0)

In [ ]:
# ── Generate predictions for both models (truth is shared) ──────────────────
preds = {}
for name, (run_dir, ckpt) in RUNS.items():
    print(f'== {name}: {Path(run_dir).name}/{ckpt} ==')
    model, ns = load_ckpt(run_dir, ckpt)
    real, gen, params, mass = generate(model, ns, test_files)
    preds[name] = dict(gen=gen, real=real, params=params, mass=mass)
    del model; torch.cuda.empty_cache()
truth = preds['baseline']['real']   # identical across models
print('shapes:', {k: v['gen'].shape for k, v in preds.items()})

## 1. High-k: per-channel patch transfer T(k) = √(P_gen / P_truth)
The headline test. If the core fine-tune worked, **core (blue) sits above baseline (orange), closer to 1** at high k — especially Stars and Total.

In [ ]:
ks, Tt = {}, {}
chans = CHANNEL_NAMES + ['Total']
def chan_field(arr, ci):
    return arr[:, ci] if ci < 3 else arr[:, :3].sum(1)
truth_pk = {}
for ci, cn in enumerate(chans):
    k, pt = stacked_pk(chan_field(truth, ci)); truth_pk[cn] = (k, pt); ks[cn] = k
fig, ax = plt.subplots(1, len(chans), figsize=(5*len(chans), 4.2))
for ci, cn in enumerate(chans):
    k, pt = truth_pk[cn]
    for name in ['baseline', 'core']:
        _, pg = stacked_pk(chan_field(preds[name]['gen'], ci))
        ax[ci].plot(k, np.sqrt(pg/pt), color=COLORS[name], label=name, lw=1.8)
    ax[ci].axhline(1, color='k', ls=':'); ax[ci].set_xscale('log')
    ax[ci].set_title(f'{cn}  T(k)'); ax[ci].set_xlabel('k [h/Mpc]'); ax[ci].set_ylim(0.5, 1.25)
ax[0].set_ylabel('T(k) = sqrt(P_gen/P_truth)'); ax[0].legend()
plt.tight_layout(); plt.show()

## 2. Core sharpness — BCG-centered radial profile & central concentration
Patches are halo-centered, so the image-centered radial profile is the BCG profile. Core should rise toward truth near r=0.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
for j, (ci, cn) in enumerate([(2, 'Stars'), (0, 'DM_hydro')]):
    for name, arr in [('truth', truth), ('baseline', preds['baseline']['gen']), ('core', preds['core']['gen'])]:
        profs = [radial_profile(f, n_bins=40, logspace=True)[1] for f in chan_field(arr, ci)]
        r, _ = radial_profile(chan_field(truth, ci)[0], n_bins=40, logspace=True)
        ax[j].plot(r*PIX, np.mean(profs, 0), color=COLORS[name], label=name, lw=1.8)
    ax[j].set_xscale('log'); ax[j].set_yscale('log'); ax[j].set_title(f'{cn} BCG profile')
    ax[j].set_xlabel('r [Mpc/h]'); ax[j].legend()
# central concentration: ratio of central (r<2px) to total mass per patch
for name, arr in [('truth', truth), ('baseline', preds['baseline']['gen']), ('core', preds['core']['gen'])]:
    s = chan_field(arr, 2); c = s[:, 62:66, 62:66].sum((1, 2)) / (s.sum((1, 2)) + 1e-30)
    ax[2].hist(c, bins=np.linspace(0, 1, 40), histtype='step', color=COLORS[name], label=name, density=True)
ax[2].set_title('Stars central concentration (r<2px / total)'); ax[2].set_xlabel('frac'); ax[2].legend()
plt.tight_layout(); plt.show()

## 3. Goal 1 — mass conservation (per-halo integrated mass, gen vs truth)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
for ci, cn in enumerate(CHANNEL_NAMES):
    mt = truth[:, ci].sum((1, 2))
    for name in ['baseline', 'core']:
        mg = preds[name]['gen'][:, ci].sum((1, 2))
        med = np.median(mg / np.clip(mt, 1e-30, None))
        ax[ci].scatter(mt, mg, s=4, alpha=0.3, color=COLORS[name], label=f'{name} (med ratio {med:.3f})')
    lim = [mt[mt>0].min(), mt.max()]; ax[ci].plot(lim, lim, 'k--', lw=1)
    ax[ci].set_xscale('log'); ax[ci].set_yscale('log'); ax[ci].set_title(f'{cn} mass'); ax[ci].legend(fontsize=8)
    ax[ci].set_xlabel('truth'); ax[ci].set_ylabel('gen')
plt.tight_layout(); plt.show()

## 4. Goal 3 — pixel-value distributions + KS (did the bulk degrade?)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
for ci, cn in enumerate(CHANNEL_NAMES):
    lt = np.log10(1 + np.clip(truth[:, ci].ravel(), 0, None))
    bins = np.linspace(0, lt.max(), 80)
    ax[ci].hist(lt, bins=bins, histtype='step', color='k', density=True, log=True, label='truth')
    for name in ['baseline', 'core']:
        lg = np.log10(1 + np.clip(preds[name]['gen'][:, ci].ravel(), 0, None))
        ks = stats.ks_2samp(lg[::13], lt[::13]).statistic
        ax[ci].hist(lg, bins=bins, histtype='step', color=COLORS[name], density=True, log=True,
                    label=f'{name} (KS {ks:.3f})')
    ax[ci].set_title(f'{cn} pixel PDF'); ax[ci].set_xlabel('log10(1+x)'); ax[ci].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Visual — stellar cores: truth / baseline / core

In [ ]:
idx = np.argsort(-truth[:, 2].sum((1, 2)))[:5]   # most massive stellar patches
fig, ax = plt.subplots(3, len(idx), figsize=(3*len(idx), 9))
for col, i in enumerate(idx):
    for row, (name, arr) in enumerate([('truth', truth), ('baseline', preds['baseline']['gen']),
                                       ('core', preds['core']['gen'])]):
        s = arr[i, 2]; vmax = np.log10(1 + truth[i, 2]).max()
        ax[row, col].imshow(np.log10(1 + s)[44:84, 44:84], vmin=0, vmax=vmax, cmap='magma')
        ax[row, col].set_xticks([]); ax[row, col].set_yticks([])
        if col == 0: ax[row, col].set_ylabel(name, fontsize=12)
fig.suptitle('Stars (log10(1+x), central 40px) — sharper core in `core` should match truth'); plt.tight_layout(); plt.show()

## 6. Scorecard

In [ ]:
print(f"{'metric':<28}{'baseline':>12}{'core':>12}{'truth':>12}")
def Tk(name, ci, kq):
    k, pt = truth_pk[chans[ci]]; _, pg = stacked_pk(chan_field(preds[name]['gen'], ci))
    return np.sqrt(pg/pt)[np.argmin(abs(k-kq))]
for ci, cn in [(2, 'Stars'), (3, 'Total')]:
    for kq in [30, 50]:
        print(f'{cn} T(k={kq})'.ljust(28) + f'{Tk("baseline",ci,kq):>12.3f}{Tk("core",ci,kq):>12.3f}{1.0:>12.3f}')
for ci, cn in enumerate(CHANNEL_NAMES):
    mt = truth[:, ci].sum((1, 2))
    for name in ['baseline', 'core']:
        r = np.median(preds[name]['gen'][:, ci].sum((1, 2)) / np.clip(mt, 1e-30, None))
        print(f'{cn} mass ratio ({name})'.ljust(28) + f'{r:>12.3f}')
for name in ['truth', 'baseline', 'core']:
    arr = truth if name == 'truth' else preds[name]['gen']
    s = arr[:, 2]; c = np.median(s[:, 62:66, 62:66].sum((1, 2)) / (s.sum((1, 2)) + 1e-30))
    print(f'Stars central conc ({name})'.ljust(28) + f'{c:>12.3f}')